<a href="https://colab.research.google.com/github/FaizaIrfan0/mini_gpt_from_scratch/blob/main/mini_GPT_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#loading the dataset
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r") as f:
    text = f.read()

    print(len(text),"\n \n")
    print(text[:500])

1115394 
 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
# building vocabulary and tokenisation
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(chars)
print(vocab_size)
char2idx={ch:i for i,ch in enumerate(chars)}
idx2char={i:ch for i,ch in enumerate(chars)}
print(char2idx)
print(idx2char)


['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11

In [ ]:
import torch
import torch.nn as nn
encode=lambda s:[char2idx[c] for c in s]
decode=lambda l:"".join([idx2char[i] for i in l])
string= "I am a girl"
enc_str=encode(string)
print(enc_str)
print(decode(enc_str))


[21, 1, 39, 51, 1, 39, 1, 45, 47, 56, 50]
I am a girl


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
k=len(data)
data_test =data[:int(0.9*k)]
data_val =data[int(0.9*k):]

In [ ]:
#GPT end-to-end arcitecture

import torch
import torch.nn as nn
import math

def add_and_layernorm(x, sub_layer):
  x+=x+sub_layer
  return nn.LayerNorm(x.size())

def masked_self_attention(q, k, v, mask=None):
  d_k = q.size(-1)
  scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
  if mask is not None:
    t = scores.size(-1)  # actual seq_len of THIS forward pass
    mask = torch.tril(torch.ones(t, t, device=scores.device))
    mask = mask.unsqueeze(0).unsqueeze(0)  # (1,1,t,t)
    scores = scores.masked_fill(mask == 0, float('-inf'))
  attn_weights = torch.softmax(scores, dim=-1)
  output = torch.matmul(attn_weights, v)
  return output, attn_weights

class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, d_model)
        )
  def forward(self, x):
    return self.net(x)


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super().__init__()
    assert d_model % num_heads == 0

    self.num_heads = num_heads
    self.d_k = d_model // num_heads
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def split_heads(self, x, batch_size):
    x = x.view(batch_size, -1, self.num_heads, self.d_k)
    return x.transpose(-2,-1)
  def forward(self, x, mask=None):
    batch_size = x.size(0)
    q = self.split_heads(self.W_q(x), batch_size)
    k = self.split_heads(self.W_k(x), batch_size)
    v = self.split_heads(self.W_v(x), batch_size)

    attn_output, attn_weights = masked_self_attention(q, k, v, mask)
    attn_output = attn_output.transpose(1, 2).contiguous()
    attn_output = attn_output.view(batch_size, -1, self.num_heads * self.d_k)
    return self.W_o(attn_output)

In [ ]:

class GPTBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.attn = MultiHeadAttention(d_model, num_heads)
    self.ff   = FeedForward(d_model, d_ff)

    self.ln1  = nn.LayerNorm(d_model)
    self.ln2  = nn.LayerNorm(d_model)

    self.drop = nn.Dropout(dropout)

  def forward(self, x, mask=None):
    x = x + self.drop(self.attn(self.ln1(x), mask))
    x = x + self.drop(self.ff(self.ln2(x)))
    return x



In [ ]:
class MiniGPT(nn.Module):
  def __init__(self, vocab_size, d_model, num_heads,num_layers, d_ff, max_seq_len, dropout=0.1):
    super().__init__()
    self.token_emb = nn.Embedding(vocab_size, d_model)
    self.pos_emb   = nn.Embedding(max_seq_len, d_model)
    self.drop      = nn.Dropout(dropout)
    self.blocks    = nn.ModuleList([
        GPTBlock(d_model, num_heads, d_ff, dropout)
        for _ in range(num_layers)
        ])
    self.ln_final  = nn.LayerNorm(d_model)
    self.lm_head   = nn.Linear(d_model, vocab_size)
    self.lm_head.weight = self.token_emb.weight
    self._init_weights()


  def _init_weights(self):
    for module in self.modules():
      if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:      # Linear CAN have bias
          nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
                                                                                        # No bias check here — Embedding has no bias

  def make_causal_mask(self, seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)
  def forward(self, idx):
    batch, seq_len = idx.size()
    device = idx.device
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    x = self.drop(self.token_emb(idx) + self.pos_emb(positions))
    mask = self.make_causal_mask(seq_len, device)
    for block in self.blocks:
      x = block(x, mask)
      x = self.ln_final(x)
      logits = self.lm_head(x)
      return logits

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

BATCH_SIZE    = 64
BLOCK_SIZE    = 128      # co
MAX_ITERS     = 15000
EVAL_INTERVAL = 500
LEARNING_RATE = 3e-4
WARMUP_ITERS  = 100
EVAL_ITERS    = 50

Using: cuda


In [ ]:
def get_batch(split, train_data, val_data):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
  x = torch.stack([data[i   : i+BLOCK_SIZE  ] for i in ix])
  y = torch.stack([data[i+1 : i+BLOCK_SIZE+1] for i in ix])
  return x.to(device), y.to(device)

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, val_data):
  model.eval()
  losses = {}
  for split in ["train", "val"]:
    split_losses = torch.zeros(EVAL_ITERS)
    for k in range(EVAL_ITERS):
      x, y = get_batch(split, train_data, val_data)
      logits = model(x)
      B, T, C = logits.shape
      loss = nn.CrossEntropyLoss()(
          logits.view(B*T, C),
          y.view(B*T)
          )
      split_losses[k] = loss.item()
    losses[split] = split_losses.mean()
  model.train()
  return losses

In [ ]:
def train(model, train_data, val_data):
      model = model.to(device)
      optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
      scheduler = CosineAnnealingLR(optimizer, T_max=MAX_ITERS, eta_min=1e-5)

      def get_lr(step):
        if step < WARMUP_ITERS:
          return LEARNING_RATE * step / WARMUP_ITERS
        return LEARNING_RATE
      print("Starting training...\n")

      for step in range(MAX_ITERS):
        lr = get_lr(step)
        for param_group in optimizer.param_groups:
          param_group['lr'] = lr
        if step % EVAL_INTERVAL == 0:
          losses = estimate_loss(model, train_data, val_data)
          print(f"Step {step:5d} | train loss: {losses['train']:.4f} | val loss: {losses['val']:.4f} | lr: {lr:.6f}")
        x, y = get_batch("train", train_data, val_data)
        logits = model(x)

        B, T, C = logits.shape
        loss = nn.CrossEntropyLoss()(
            logits.view(B*T, C),
            y.view(B*T)
            )

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
      print("\nTraining complete!")
      return model

In [ ]:
model = MiniGPT(
    vocab_size  = 65,   # GPT-2 vocab size (BPE)
    d_model     = 256,     # small for mini version
    num_heads   = 8,
    num_layers  = 6,
    d_ff        = 1024,    # 4 * d_model
    max_seq_len = 128,
    dropout     = 0.1
                              )
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 4,788,545


In [ ]:
trained_model = train(model, data_test, data_val)

Starting training...

Step     0 | train loss: 4.2315 | val loss: 4.2262 | lr: 0.000000
Step   500 | train loss: 1.9297 | val loss: 1.9487 | lr: 0.000300
Step  1000 | train loss: 1.8666 | val loss: 1.8858 | lr: 0.000300
Step  1500 | train loss: 1.8475 | val loss: 1.8687 | lr: 0.000300
Step  2000 | train loss: 1.8380 | val loss: 1.8566 | lr: 0.000300
Step  2500 | train loss: 1.8350 | val loss: 1.8612 | lr: 0.000300
Step  3000 | train loss: 1.8276 | val loss: 1.8493 | lr: 0.000300
Step  3500 | train loss: 1.8263 | val loss: 1.8502 | lr: 0.000300
Step  4000 | train loss: 1.8222 | val loss: 1.8468 | lr: 0.000300
Step  4500 | train loss: 1.8204 | val loss: 1.8413 | lr: 0.000300
Step  5000 | train loss: 1.8178 | val loss: 1.8449 | lr: 0.000300
Step  5500 | train loss: 1.8180 | val loss: 1.8366 | lr: 0.000300
Step  6000 | train loss: 1.8110 | val loss: 1.8416 | lr: 0.000300
Step  6500 | train loss: 1.8111 | val loss: 1.8382 | lr: 0.000300
Step  7000 | train loss: 1.8117 | val loss: 1.8360 | l

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=500, temperature=0.8):
  model.eval()
  device = next(model.parameters()).device
  idx = torch.tensor(encode(prompt), dtype=torch.long)
  idx = idx.unsqueeze(0).to(device)  # (1, seq_len)
  for _ in range(max_new_tokens):
    idx_cond = idx[:, -BLOCK_SIZE:]
    logits = model(idx_cond)
    logits = logits[:, -1, :] / temperature
    probs = torch.softmax(logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    idx = torch.cat([idx, next_token], dim=1)
  return decode(idx[0].tolist())
output = generate(trained_model, prompt='you ', max_new_tokens=300)
print(output)

you male,
Alo thixthexts Ris me cilond; ng, ho be WI be ONom t; cewir'd--
Tze l?

TZAne;
Pengh ig om?
G IARETIShlle llin he f sinf I grace nost to and thee she have peny but I have som
Ore have it strain be cone were I um nome bort in hand,
Them price hath the he granch up; the he wire with no the crowb


in this notebook i have demonstrated an end to end implementatiom of generative pretrained model as published by open ai in 2020(whatever it is) in a mini scale. the number of parameters were effectively scaled down to ease out the training process on a mcuh smaller scale. tiny shakespere dataset was used for trainign purposes.
i began by implementing various components of gpt architecture seperately, including add and layer norm, multi head attention coupled with masked head attemtion, which is quintessential to gpt, feef forward layer. then everything was stacied into a gptBlock class amd it was further iterated on miniGPT class. traing loops, backprop and optimizers were implemented. finally, hyperparamerters were passed and model gave training error of 1.78 and test error of 1.82 after 15000 iterations at learning rate of 0.0003 and using Adam optimizer.